# Config

In [1]:
!pip install sentence-transformers==5.1.0

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [2]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


In [3]:
import pandas as pd
import os
from preprocess.preprocess import clean_text, check_deleted_expressions
from preprocess.translate import translator, gen_text_for_embedding, final_clean, detect_language
import time
import json
import numpy as np

# 4) TF ID feature extractor

## Train

In [22]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat_summary.csv")
df = pd.read_csv(filepath)

#Prueba con solo textos traducidos
#df = df[df["Español"]==False]

In [23]:
from sklearn.preprocessing import LabelEncoder
from models.TIFD import gen_TFID_vectors
import numpy as np
from utils.dataset import gen_dataset_select_cols

#Detectar textos en español
cols = {
    "Resumen": "Resumen_process",
}
#Preprocesamiento de datos
df[list(cols.values())] = df[list(cols.keys())].applymap(clean_text)
df["Español"]=detect_language(df["Resumen_process"])

#Columnas a seleccionar para clasificación
cols = ["Titulo_trad", "keywords_trad", "summary_100_words"]

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset_select_cols(codes_test, df, cols = cols)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset_select_cols(codes_train, df, cols = cols)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)
#Creacion de vectores TFID
X_train, X_test = gen_TFID_vectors(X_train, X_test)
print(X_train.shape, X_test.shape)

(771, 8006) (193, 8006)


In [24]:
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model, eval_model, mlflow_ckeckpoint
from utils.dataset import CvCustom
from collections import Counter

# 2. Elegir modelos a probar
model_keys = [
    'LogisticRegression',
    #'DecisionTreeClassifier',
    'RandomForestClassifier',
    #'GradientBoostingClassifier',
    'XGBClassifier',
    #'MLPClassifier',
    'SVC',
    #'SGDClassifier'
]

# 3. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)

print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))
# 4. Ejecutar entrenamiento, validación y test con tus funciones
n_iter=20
sample_weight_On=True
scoring='f1_macro'
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode), 
                                                 n_iter=n_iter, sample_weight_On = sample_weight_On)

best_model = select_best_model(results_val, models_dicc)

# 5. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

📊 train: Counter({1: 444, 0: 327})
📊 test: Counter({1: 111, 0: 82})
(771,)
LogisticRegression
Compute sw


/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [10, 'l2', 'lbfgs'] before, using random point [0.1, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [1.0, 'l2', 'lbfgs'] before, using random point [1.0, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [1.0, 'l2', 'lbfgs'] before, using random point [0.1, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [1.0, 'l2', 'lbfgs'] before, using random point [10, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [1.0, 'l2', '

RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.6, 'std_test_score': 0.03}
RandomForestClassifier: {'mean_test_score': 0.63, 'std_test_score': 0.02}
XGBClassifier: {'mean_test_score': 0.64, 'std_test_score': 0.01}
SVC: {'mean_test_score': 0.6, 'std_test_score': 0.02}


In [25]:
# Métricas por idioma
lang_es = df_test["Español"]
for name, model in models_dicc.items():
    print(name)
    results=eval_model(model, X_test, y_test, lang_es)
    print(results)

LogisticRegression
{'accuracy': 0.694300518134715, 'f1_macro': 0.6867314093922805, 'cm': array([[52, 30],
       [29, 82]]), 'f1_es': 0.6751475293674782, 'f1_en': 0.7171990799897776, 'cm_es': array([[20, 20],
       [17, 58]]), 'cm_en': array([[32, 10],
       [12, 24]]), 'precision': 0.7321428571428571, 'recall': 0.7387387387387387}
RandomForestClassifier
{'accuracy': 0.5803108808290155, 'f1_macro': 0.5766283006093431, 'cm': array([[47, 35],
       [46, 65]]), 'f1_es': 0.5873495643316615, 'f1_en': 0.5426297840090943, 'cm_es': array([[15, 25],
       [22, 53]]), 'cm_en': array([[32, 10],
       [24, 12]]), 'precision': 0.65, 'recall': 0.5855855855855856}
XGBClassifier
{'accuracy': 0.5699481865284974, 'f1_macro': 0.5631936301911489, 'cm': array([[43, 39],
       [44, 67]]), 'f1_es': 0.5900709633447172, 'f1_en': 0.5328671328671328, 'cm_es': array([[16, 24],
       [23, 52]]), 'cm_en': array([[27, 15],
       [21, 15]]), 'precision': 0.6320754716981132, 'recall': 0.6036036036036037}
SVC
{

## Save

In [27]:
exp_info = {
    'exp_name': "Bayesiansearchcv_summary",
    #'artifact_path': "file:///tmp/mlflow_experiments/mlruns", 
    #'tracking_path': "sqlite:////tmp/mlflow_experiments/mlflow.db",
}

extra_parms = {
    "n_iter": n_iter,
    "sample_weight_On": sample_weight_On,
    "scoring": scoring,
    "cols": cols,
    "Features": "TF-IDF",
}

mlflow_ckeckpoint(exp_info, results_val, models_dicc, extra_parms, X_test, y_test, lang_es)

Current tracking uri: http://mlflow-server:5000
📝 Registrando modelo en MLflow: LogisticRegression


2025/09/09 19:46:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run LogisticRegression at: http://mlflow-server:5000/#/experiments/13/runs/5fbf840e17434503905bcc9d791f5195
🧪 View experiment at: http://mlflow-server:5000/#/experiments/13
📝 Registrando modelo en MLflow: RandomForestClassifier


2025/09/09 19:46:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run RandomForestClassifier at: http://mlflow-server:5000/#/experiments/13/runs/da0a16127e014cb58bf641f112633fe5
🧪 View experiment at: http://mlflow-server:5000/#/experiments/13
📝 Registrando modelo en MLflow: XGBClassifier


2025/09/09 19:46:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://mlflow-server:5000/#/experiments/13/runs/02ccecd0de5349e8bea4467a7b34d2dd
🧪 View experiment at: http://mlflow-server:5000/#/experiments/13
📝 Registrando modelo en MLflow: SVC


2025/09/09 19:46:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run SVC at: http://mlflow-server:5000/#/experiments/13/runs/a2a47653ceba4389b3156baa992aa315
🧪 View experiment at: http://mlflow-server:5000/#/experiments/13


# 5) SPECTER model

In [28]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat_summary.csv")
df = pd.read_csv(filepath)

In [29]:
#del gen_dataset
from utils.dataset import gen_dataset_select_cols
from models.specter import embed_texts
import numpy as np
from sklearn.preprocessing import LabelEncoder

#Detectar textos en español
cols = {
    "Resumen": "Resumen_process",
}
#Preprocesamiento de datos
df[list(cols.values())] = df[list(cols.keys())].applymap(clean_text)
df["Español"]=detect_language(df["Resumen_process"])

#Columnas a seleccionar para clasificación
cols = ["Titulo_trad", "keywords_trad", "summary_100_words"]
element_names=["Title:", "keywords:", "abstract:"]

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset_select_cols(codes_test, df, cols = cols, element_names=element_names)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset_select_cols(codes_train, df, cols = cols)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# 2) Calcular embeddings
# Parámetros modelo
BASE_MODEL = "allenai/specter2_base"
#ADAPTER_NAME = "allenai/specter2"
ADAPTER_NAME="allenai/specter2_classification"
X_train = embed_texts(X_train, BASE_MODEL, ADAPTER_NAME)
X_test = embed_texts(X_test, BASE_MODEL, ADAPTER_NAME)

print(X_train.shape, X_test.shape)

/usr/local/lib/python3.10/dist-packages/torch/_utils.py:830: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.
(771, 768) (193, 768)


In [34]:
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model, eval_model, mlflow_ckeckpoint
from utils.dataset import CvCustom
from collections import Counter

# 2. Elegir modelos a probar
model_keys = [
    #'LogisticRegression',
    #'DecisionTreeClassifier',
    #'RandomForestClassifier',
    #'GradientBoostingClassifier',
    #'XGBClassifier',
    #'MLPClassifier',
    'SVC',
    #'SGDClassifier'
]

# 3. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)

print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))
# 4. Ejecutar entrenamiento, validación y test con tus funciones
n_iter=20
sample_weight_On=True
scoring="f1_macro"
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode), 
                                                 n_iter=n_iter, sample_weight_On = sample_weight_On)

best_model = select_best_model(results_val, models_dicc)

# 5. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

📊 train: Counter({1: 444, 0: 327})
📊 test: Counter({1: 111, 0: 82})
(771,)
SVC
Compute sw

🔍 Validación:
SVC: {'mean_test_score': 0.64, 'std_test_score': 0.03}


In [35]:
# Métricas por idioma
lang_es = df_test["Español"]
for name, model in models_dicc.items():
    print(name)
    results=eval_model(model, X_test, y_test, lang_es)
    print(results)

SVC
{'accuracy': 0.6424870466321243, 'f1_macro': 0.636871813050473, 'cm': array([[50, 32],
       [37, 74]]), 'f1_es': 0.6271403199717567, 'f1_en': 0.6644385026737968, 'cm_es': array([[19, 21],
       [22, 53]]), 'cm_en': array([[31, 11],
       [15, 21]]), 'precision': 0.6981132075471698, 'recall': 0.6666666666666666}


## Save

In [36]:
exp_info = {
    'exp_name': "Bayesiansearchcv_summary",
    #'artifact_path': "file:///tmp/mlflow_experiments/mlruns", 
    #'tracking_path': "sqlite:////tmp/mlflow_experiments/mlflow.db",
}

extra_parms = {
    "n_iter": n_iter,
    "sample_weight_On": sample_weight_On,
    "scoring": scoring,
    "cols": cols,
    "Features": "SPECTER2",
}

mlflow_ckeckpoint(exp_info, results_val, models_dicc, extra_parms, X_test, y_test, lang_es)

Current tracking uri: http://mlflow-server:5000


📝 Registrando modelo en MLflow: SVC


2025/09/09 19:55:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run SVC at: http://mlflow-server:5000/#/experiments/13/runs/d8c9851a538644c1afce7406c837e4ed
🧪 View experiment at: http://mlflow-server:5000/#/experiments/13


# 6) all-roberta-large-v1

## Train

In [18]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

In [20]:
#del gen_dataset
from utils.dataset import gen_dataset_select_cols
import numpy as np
from sklearn.preprocessing import LabelEncoder

#Columnas a seleccionar para clasificación
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]
element_names=["Title:", "keywords:", "abstract:"]

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset_select_cols(codes_test, df, cols = cols, element_names=element_names)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset_select_cols(codes_train, df, cols = cols)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [21]:
from transformers import RobertaTokenizerFast, RobertaModel
import torch
import warnings
from tqdm import tqdm
warnings.filterwarnings("ignore", message="Some weights of the model.*were not initialized.*")

def roberta_encoder_batch(texts, batch_size=8, max_length=512):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model_name = "roberta-large"
    tokenizer = RobertaTokenizerFast.from_pretrained(model_name)
    model = RobertaModel.from_pretrained(model_name).to(device)

    model.eval()  # desactiva dropout
    embeddings = []

    # recorrer en lotes de batch_size
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]

        # tokenización por lote
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        ).to(device) 

        with torch.no_grad():
            outputs = model(**inputs)

        # embeddings del token <s> ([CLS]) para cada texto del batch
        cls_embeddings = outputs.last_hidden_state[:, 0, :]  # (batch, hidden_dim)
        embeddings.append(cls_embeddings.cpu().numpy())

    # concatenar todos los batches
    return np.vstack(embeddings)  # (n_texts, hidden_dim)

In [22]:
X_train = roberta_encoder_batch(X_train)
X_test = roberta_encoder_batch(X_test)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
print(X_train.shape, X_test.shape)

(771, 1024) (193, 1024)


In [24]:
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model, eval_model, mlflow_ckeckpoint
from utils.dataset import CvCustom
from collections import Counter

# 2. Elegir modelos a probar
model_keys = [
    'LogisticRegression',
    #'DecisionTreeClassifier',
    'RandomForestClassifier',
    #'GradientBoostingClassifier',
    'XGBClassifier',
    #'MLPClassifier',
    'SVC',
    #'SGDClassifier'
]

# 3. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)

print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))
# 4. Ejecutar entrenamiento, validación y test con tus funciones
n_iter=20
sample_weight_On=True
scoring="f1_macro"
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode), 
                                                 n_iter=n_iter, sample_weight_On = sample_weight_On)

best_model = select_best_model(results_val, models_dicc)

# 5. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

📊 train: Counter({1: 444, 0: 327})
📊 test: Counter({1: 111, 0: 82})
(771,)
LogisticRegression
Compute sw


/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.61, 'std_test_score': 0.01}
RandomForestClassifier: {'mean_test_score': 0.62, 'std_test_score': 0.02}
XGBClassifier: {'mean_test_score': 0.62, 'std_test_score': 0.03}
SVC: {'mean_test_score': 0.6, 'std_test_score': 0.01}


In [25]:
# Métricas por idioma
lang_es = df_test["Español"]
for name, model in models_dicc.items():
    print(name)
    results=eval_model(model, X_test, y_test, lang_es)
    print(results)

LogisticRegression
{'accuracy': 0.6476683937823834, 'precision': 0.7087378640776699, 'recall': 0.6576576576576577, 'f1_macro': 0.6434470767224516, 'cm': array([[52, 30],
       [38, 73]]), 'f1_es': 0.6772899073139264, 'f1_en': 0.5934065934065934, 'cm_es': array([[21, 19],
       [18, 57]]), 'cm_en': array([[31, 11],
       [20, 16]])}
RandomForestClassifier
{'accuracy': 0.6217616580310881, 'precision': 0.6637931034482759, 'recall': 0.6936936936936937, 'f1_macro': 0.6096473000304768, 'cm': array([[43, 39],
       [34, 77]]), 'f1_es': 0.645422630299757, 'f1_en': 0.5588189588189588, 'cm_es': array([[15, 25],
       [14, 61]]), 'cm_en': array([[28, 14],
       [20, 16]])}
XGBClassifier
{'accuracy': 0.6528497409326425, 'precision': 0.6833333333333333, 'recall': 0.7387387387387387, 'f1_macro': 0.6388493227202905, 'cm': array([[44, 38],
       [29, 82]]), 'f1_es': 0.6878449790025153, 'f1_en': 0.5902844683332489, 'cm_es': array([[19, 21],
       [14, 61]]), 'cm_en': array([[25, 17],
       [15

## Save

In [26]:
exp_info = {
    'exp_name': "Bayesiansearchcv_all-roberta-large-v1_department",
    #'artifact_path': "file:///tmp/mlflow_experiments/mlruns", 
    #'tracking_path': "sqlite:////tmp/mlflow_experiments/mlflow.db",
}

extra_parms = {
    "n_iter": n_iter,
    "sample_weight_On": sample_weight_On,
    "scoring": scoring,
    "cols": cols
}

mlflow_ckeckpoint(exp_info, results_val, models_dicc, extra_parms, X_test, y_test, lang_es)

Current tracking uri: http://mlflow-server:5000
📝 Registrando modelo en MLflow: LogisticRegression


2025/09/03 21:10:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run LogisticRegression at: http://mlflow-server:5000/#/experiments/9/runs/dbf1e4cbe8ba49b6829926d261b33215
🧪 View experiment at: http://mlflow-server:5000/#/experiments/9
📝 Registrando modelo en MLflow: RandomForestClassifier


2025/09/03 21:10:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run RandomForestClassifier at: http://mlflow-server:5000/#/experiments/9/runs/360ac5f7d36f4bf689846dde28d62e05
🧪 View experiment at: http://mlflow-server:5000/#/experiments/9
📝 Registrando modelo en MLflow: XGBClassifier


2025/09/03 21:10:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://mlflow-server:5000/#/experiments/9/runs/074ac708697f428ba5dc9e407f0db697
🧪 View experiment at: http://mlflow-server:5000/#/experiments/9
📝 Registrando modelo en MLflow: SVC


2025/09/03 21:11:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run SVC at: http://mlflow-server:5000/#/experiments/9/runs/355c83710baf4bdd84b61d2bb5b89dcd
🧪 View experiment at: http://mlflow-server:5000/#/experiments/9


# 7) all-mpnet-base-v2 

## Train

In [28]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

In [29]:
#del gen_dataset
from utils.dataset import gen_dataset_select_cols
import numpy as np
from sklearn.preprocessing import LabelEncoder

#Columnas a seleccionar para clasificación
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]
element_names=["Title:", "keywords:", "abstract:"]

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset_select_cols(codes_test, df, cols = cols, element_names=element_names)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset_select_cols(codes_train, df, cols = cols)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [ ]:
from sentence_transformers import SentenceTransformer
import torch
import numpy as np

def encoder_batch_sentence_transformers(model_name, texts, batch_size=8, max_length=512):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SentenceTransformer(model_name, device=device)  # ya queda en GPU si existe

    # encode hace batching automáticamente, pero si quieres controlar batch_size lo pasas como argumento
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        show_progress_bar=True
    )

    return embeddings  # (n_texts, hidden_dim)

In [ ]:
model_name="all-mpnet-base-v2"
X_train = roberta_encoder_batch(model_name, X_train)
X_test = roberta_encoder_batch(model_name, X_test)

print(X_train.shape, X_test.shape)

In [32]:
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model, eval_model, mlflow_ckeckpoint
from utils.dataset import CvCustom
from collections import Counter

# 2. Elegir modelos a probar
model_keys = [
    'LogisticRegression',
    #'DecisionTreeClassifier',
    'RandomForestClassifier',
    #'GradientBoostingClassifier',
    'XGBClassifier',
    #'MLPClassifier',
    'SVC',
    #'SGDClassifier'
]

# 3. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)

print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))
# 4. Ejecutar entrenamiento, validación y test con tus funciones
n_iter=20
sample_weight_On=True
scoring="f1_macro"
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode), 
                                                 n_iter=n_iter, sample_weight_On = sample_weight_On)

best_model = select_best_model(results_val, models_dicc)

# 5. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

📊 train: Counter({1: 444, 0: 327})
📊 test: Counter({1: 111, 0: 82})
(771,)
LogisticRegression
Compute sw


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.62, 'std_test_score': 0.01}
RandomForestClassifier: {'mean_test_score': 0.62, 'std_test_score': 0.01}
XGBClassifier: {'mean_test_score': 0.61, 'std_test_score': 0.02}
SVC: {'mean_test_score': 0.62, 'std_test_score': 0.01}


In [33]:
# Métricas por idioma
lang_es = df_test["Español"]
for name, model in models_dicc.items():
    print(name)
    results=eval_model(model, X_test, y_test, lang_es)
    print(results)

LogisticRegression
{'accuracy': 0.6269430051813472, 'precision': 0.696969696969697, 'recall': 0.6216216216216216, 'f1_macro': 0.624025974025974, 'cm': array([[52, 30],
       [42, 69]]), 'f1_es': 0.5772655840754322, 'f1_en': 0.6983339241403759, 'cm_es': array([[17, 23],
       [26, 49]]), 'cm_en': array([[35,  7],
       [16, 20]])}
RandomForestClassifier
{'accuracy': 0.6839378238341969, 'precision': 0.7358490566037735, 'recall': 0.7027027027027027, 'f1_macro': 0.6789736318272298, 'cm': array([[54, 28],
       [33, 78]]), 'f1_es': 0.6424023297262426, 'f1_en': 0.7404817404817405, 'cm_es': array([[19, 21],
       [20, 55]]), 'cm_en': array([[35,  7],
       [13, 23]])}
XGBClassifier
{'accuracy': 0.689119170984456, 'precision': 0.7107438016528925, 'recall': 0.7747747747747747, 'f1_macro': 0.6758844603672189, 'cm': array([[47, 35],
       [25, 86]]), 'f1_es': 0.6817895400126024, 'f1_en': 0.6791154164807852, 'cm_es': array([[17, 23],
       [12, 63]]), 'cm_en': array([[30, 12],
       [13, 

## Save

In [34]:
exp_info = {
    'exp_name': "Bayesiansearchcv_all-mpnet-base-v2_department",
    #'artifact_path': "file:///tmp/mlflow_experiments/mlruns", 
    #'tracking_path': "sqlite:////tmp/mlflow_experiments/mlflow.db",
}

extra_parms = {
    "n_iter": n_iter,
    "sample_weight_On": sample_weight_On,
    "scoring": scoring,
    "cols": cols
}

mlflow_ckeckpoint(exp_info, results_val, models_dicc, extra_parms, X_test, y_test, lang_es)

2025/09/03 21:28:10 INFO mlflow.tracking.fluent: Experiment with name 'Bayesiansearchcv_all-mpnet-base-v2_department' does not exist. Creating a new experiment.


Current tracking uri: http://mlflow-server:5000
📝 Registrando modelo en MLflow: LogisticRegression


2025/09/03 21:28:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run LogisticRegression at: http://mlflow-server:5000/#/experiments/10/runs/3f3eda16f6424797ade2f6891101814e
🧪 View experiment at: http://mlflow-server:5000/#/experiments/10
📝 Registrando modelo en MLflow: RandomForestClassifier


2025/09/03 21:28:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run RandomForestClassifier at: http://mlflow-server:5000/#/experiments/10/runs/09145f1b580b409fa821d4d690aeaef8
🧪 View experiment at: http://mlflow-server:5000/#/experiments/10
📝 Registrando modelo en MLflow: XGBClassifier


2025/09/03 21:28:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://mlflow-server:5000/#/experiments/10/runs/49e3fa9f3ed64ceea2203985bd0e815e
🧪 View experiment at: http://mlflow-server:5000/#/experiments/10
📝 Registrando modelo en MLflow: SVC


2025/09/03 21:28:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run SVC at: http://mlflow-server:5000/#/experiments/10/runs/bea2912656854de39f1c063162e50c5f
🧪 View experiment at: http://mlflow-server:5000/#/experiments/10
